In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression 
from sklearn.metrics import r2_score, mean_squared_error
import shap

# ---------- 0. 路径 ----------
DATA_FILE   = "Wealthy_scored.xlsx"
UNSCORED    = "images.xlsx"
OUT_DIR     = Path("D:\Desktop\output_wealthy")
OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX = OUT_DIR / "images_predicted_Wealthy.xlsx"
BSWARM_PNG   = OUT_DIR / "shap_beeswarm_Wealthy.png"
BAR_PNG      = OUT_DIR / "shap_bar_Wealthy.png"

TARGET   = "Wealthy"
FEATS    = ["Road","Building","Pole Group","Indicator","Vegetation","Sky",
            "Person","Car","Motorcycle","Bicycle","Clothes","Trash Can",
            "Riverway","Signboard","Air Conditioner Condenser","Festival Elements"]

# ---------- 1. 读取已打分数据 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET] = df[TARGET].round(5)          

X = df[FEATS].values
y = df[TARGET].values

# ---------- 2. 划分 & 训练 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=8)    

rf = RandomForestRegressor(
        n_estimators     = 500,
        max_depth        = 8,
        min_samples_leaf = 5,
        random_state     = 42,
        n_jobs           = -1
     )
rf.fit(X_tr, y_tr)

iso = IsotonicRegression(
        y_min=y.min(),                 
        y_max=y.max(),
        increasing=True,
        out_of_bounds="clip" 
     )
iso.fit(rf.predict(X), y)  

# ---------- 3. 评估 ----------
def _met(t, p): return r2_score(t, p), mean_squared_error(t, p, squared=False)

raw_tr, raw_te = rf.predict(X_tr), rf.predict(X_te)
cal_tr, cal_te = iso.transform(raw_tr), iso.transform(raw_te)

r2_tr,  rmse_tr  = _met(y_tr,  cal_tr)
r2_te,  rmse_te  = _met(y_te,  cal_te)
r2_all, rmse_all = _met(y,     iso.transform(rf.predict(X)))

print(f"Train   R² = {r2_tr :.3f} | RMSE = {rmse_tr :.3f}")
print(f"Test    R² = {r2_te :.3f} | RMSE = {rmse_te :.4f}")
print(f"Overall R² = {r2_all:.3f} | RMSE = {rmse_all:.3f}")

# ---------- 4. SHAP ----------
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(df[FEATS])

# 蜂群图
plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, df[FEATS], feature_names=FEATS, show=False)
plt.title("SHAP Summary (Beeswarm) – Wealthy Score")
plt.savefig(BSWARM_PNG, bbox_inches="tight", dpi=300); plt.close()

# 条形图
plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, df[FEATS], feature_names=FEATS,
                  plot_type="bar", show=False)
plt.title("Feature Importance (mean |SHAP value|)")
plt.savefig(BAR_PNG, bbox_inches="tight", dpi=300); plt.close()

# ---------- 5. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED)
df_new[TARGET] = rf.predict(df_new[FEATS]).round(5)
df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测结果:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)



=== RF + Isotonic (Safety) ===
Train   R² = 0.9181 | RMSE = 0.1485
Test    R² = 0.8308 | RMSE = 0.2247
Overall R² = 0.8992 | RMSE = 0.1665



d:\Anaconda\Lib\site-packages\sklearn\base.py:457: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


✔ 预测结果: D:\Desktop\output_wealthy\images_predicted_Wealthy.xlsx
✔ SHAP 图: D:\Desktop\output_wealthy\shap_beeswarm_Wealthy.png D:\Desktop\output_wealthy\shap_bar_Wealthy.png
